In [1]:
import requests
import zipfile
import io
import xml.etree.ElementTree as ET
import pandas as pd

# ==========================================
# 1. CONFIGURATION
# ==========================================
CONGRESS_NUM = "119"
BILL_TYPES = ["hconres", "hr", "hres", "hjres", "s", "sconres", "sres", "sjres"]
BASE_URL = "https://www.govinfo.gov/bulkdata/BILLSTATUS/{}/{}/BILLSTATUS-{}-{}.zip"

# ==========================================
# 2. MODIFIED EXTRACTION FUNCTION
# ==========================================
def parse_bill_cosponsors(xml_content):
    try:
        root = ET.fromstring(xml_content)
        bill = root.find('bill')

        if bill is None:
            return []

        def get_text(node, path):
            found = node.find(path)
            return found.text if found is not None else None

        # --- A. GENERAL BILL DATA ---
        latest_action_node = bill.find('actions/item')

        l_code = None
        l_text = None
        l_type = None

        if latest_action_node is not None:
            l_code = get_text(latest_action_node, 'actionCode')
            l_text = get_text(latest_action_node, 'text')
            l_type = get_text(latest_action_node, 'type')

        base_data = {
            'Congress': get_text(bill, 'congress'),
            'Number': get_text(bill, 'number'),
            'Type': get_text(bill, 'type'),
            'Origin_Chamber': get_text(bill, 'originChamber'),
            'Introduced_Date': get_text(bill, 'introducedDate'), # <--- CHANGE: column added
            'latest_action_code': l_code,
            'latest_action_type': l_type,
            'latest_action_text': l_text
        }

        # --- B. PRIMARY SPONSOR DATA ---
        sponsor_node = bill.find('sponsors/item')
        if sponsor_node is not None:
            base_data['sponsor_name'] = get_text(sponsor_node, 'fullName')
            base_data['sponsor_bioguide'] = get_text(sponsor_node, 'bioguideId')
            base_data['sponsor_party'] = get_text(sponsor_node, 'party')
        else:
            base_data.update({'sponsor_name': None, 'sponsor_bioguide': None, 'sponsor_party': None})

        # --- C. COSPONSOR DATA ---
        rows = []
        cosponsors_list = bill.findall('cosponsors/item')

        if len(cosponsors_list) > 0:
            for cosponsor in cosponsors_list:
                row = base_data.copy()
                row['cosponsor_name'] = get_text(cosponsor, 'fullName')
                row['cosponsor_bioguide'] = get_text(cosponsor, 'bioguideId')
                row['cosponsor_party'] = get_text(cosponsor, 'party')
                row['cosponsorship_date'] = get_text(cosponsor, 'sponsorshipDate')
                rows.append(row)
        else:
            row = base_data.copy()
            row.update({
                'cosponsor_name': None, 'cosponsor_bioguide': None,
                'cosponsor_party': None, 'cosponsorship_date': None
            })
            rows.append(row)

        return rows

    except Exception as e:
        print(f"Error parseando XML: {e}")
        return []

# ==========================================
# 3. MAIN PROCESS
# ==========================================
all_rows = []
print(f"Starting download and extraction for Congress {CONGRESS_NUM}...\n")

for b_type in BILL_TYPES:
    download_url = BASE_URL.format(CONGRESS_NUM, b_type, CONGRESS_NUM, b_type)
    print(f"--> Processing: {b_type.upper()}")

    try:
        response = requests.get(download_url, stream=True)
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                xml_files = [f for f in z.namelist() if f.endswith('.xml')]
                for filename in xml_files:
                    with z.open(filename) as f:
                        all_rows.extend(parse_bill_cosponsors(f.read()))
        else:
            print(f"    [!] No disponible (Status {response.status_code})")
    except Exception as e:
        print(f"    Error en ZIP {b_type}: {e}")

# ==========================================
# 4. FINAL DATAFRAME
# ==========================================
df = pd.DataFrame(all_rows)

cols_order = [
    'Congress', 'Number', 'Type', 'Origin_Chamber', 'Introduced_Date', # <--- CHANGE: included in the ordering
    'latest_action_code', 'latest_action_type', 'latest_action_text',
    'sponsor_name', 'sponsor_bioguide', 'sponsor_party',
    'cosponsor_name', 'cosponsor_bioguide', 'cosponsor_party', 'cosponsorship_date'
]
df = df[[c for c in cols_order if c in df.columns]]

print(f"\nTotal rows generated: {len(df)}")
print(df.head(10))

# df.to_csv("cosponsors_119_detailed.csv", index=False)

Iniciando descarga y extracción para el Congreso 119...

--> Procesando: HCONRES
--> Procesando: HR
--> Procesando: HRES
--> Procesando: HJRES
--> Procesando: S
--> Procesando: SCONRES
--> Procesando: SRES
--> Procesando: SJRES

Total de filas generadas: 157596
  Congress Number     Type Origin_Chamber Introduced_Date latest_action_code  \
0      119      2  HCONRES          House      2025-01-03             H11100   
1      119      1  HCONRES          House      2025-01-03               None   
2      119      3  HCONRES          House      2025-01-13             H11100   
3      119      3  HCONRES          House      2025-01-13             H11100   
4      119      3  HCONRES          House      2025-01-13             H11100   
5      119      3  HCONRES          House      2025-01-13             H11100   
6      119      3  HCONRES          House      2025-01-13             H11100   
7      119      3  HCONRES          House      2025-01-13             H11100   
8      119      3 

In [2]:
df = df.dropna(subset=['cosponsor_party'])


In [3]:
df

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13
5,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13
6,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157590,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Wyden, Ron [D-OR]",W000779,D,2026-03-02
157591,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Cantwell, Maria [D-WA]",C000127,D,2026-03-02
157592,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Booker, Cory A. [D-NJ]",B001288,D,2026-03-04
157593,119,181,SJRES,Senate,2026-04-13,None,IntroReferral,Read twice and referred to the Committee on Fo...,"Sen. Gillibrand, Kirsten E. [D-NY]",G000555,D,"Sen. Schumer, Charles E. [D-NY]",S000148,D,2026-04-16


In [4]:
democrats_list = ['S000033', 'K000383']
republicans_list = ['K000401']

df.loc[df['sponsor_bioguide'].isin(democrats_list), 'sponsor_party'] = 'D'
df.loc[df['sponsor_bioguide'].isin(republicans_list), 'sponsor_party'] = 'R'

df.loc[df['cosponsor_bioguide'].isin(democrats_list), 'cosponsor_party'] = 'D'
df.loc[df['cosponsor_bioguide'].isin(republicans_list), 'cosponsor_party'] = 'R'

In [5]:
df

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13
5,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13
6,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157590,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Wyden, Ron [D-OR]",W000779,D,2026-03-02
157591,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Cantwell, Maria [D-WA]",C000127,D,2026-03-02
157592,119,104,SJRES,Senate,2026-01-29,None,Discharge,Motion to discharge Senate Committee on Foreig...,"Sen. Kaine, Tim [D-VA]",K000384,D,"Sen. Booker, Cory A. [D-NJ]",B001288,D,2026-03-04
157593,119,181,SJRES,Senate,2026-04-13,None,IntroReferral,Read twice and referred to the Committee on Fo...,"Sen. Gillibrand, Kirsten E. [D-NY]",G000555,D,"Sen. Schumer, Charles E. [D-NY]",S000148,D,2026-04-16


In [6]:
df['bill_code'] = df['Type'] + '-' + df['Number'].astype(str)

In [7]:
# Convert sponsorshipDate to datetime if it isn't already
df['cosponsorship_date'] = pd.to_datetime(df['cosponsorship_date'], errors='coerce')
df['Introduced_Date'] = pd.to_datetime(df['Introduced_Date'], errors='coerce')
# Get the earliest (minimum) date per bill_code
first_sponsorship = df.groupby('bill_code')['cosponsorship_date'].min().reset_index()
first_sponsorship.columns = ['bill_code', 'first_cosponsorshipDate']

# Merge with the original dataframe
df_final = df.merge(first_sponsorship, on='bill_code', how='left')

# View the result
df_final.head(100)

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Vargas, Juan [D-CA-52]",V000130,D,2025-04-24,HCONRES-4,2025-01-28
96,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Thompson, Glenn [R-PA-15]",T000467,R,2025-04-24,HCONRES-4,2025-01-28
97,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Wagner, Ann [R-MO-2]",W000812,R,2025-05-05,HCONRES-4,2025-01-28
98,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Moran, Nathaniel [R-TX-1]",M001224,R,2025-05-19,HCONRES-4,2025-01-28


In [8]:
df_final['days_to_join'] = df_final['cosponsorship_date'] - df_final['Introduced_Date']
df_final['days_to_join'] = df_final['days_to_join'].dt.days
df_final.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13,0
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13,0
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13,0
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13,0
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13,0


In [9]:
df_final.head(100)

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13,0
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13,0
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13,0
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13,0
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Vargas, Juan [D-CA-52]",V000130,D,2025-04-24,HCONRES-4,2025-01-28,86
96,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Thompson, Glenn [R-PA-15]",T000467,R,2025-04-24,HCONRES-4,2025-01-28,86
97,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Wagner, Ann [R-MO-2]",W000812,R,2025-05-05,HCONRES-4,2025-01-28,97
98,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Moran, Nathaniel [R-TX-1]",M001224,R,2025-05-19,HCONRES-4,2025-01-28,111


In [10]:
df_final['days_to_join'].describe()

,days_to_join
count,154182.000000
mean,46.783172
std,92.361058
min,0.000000
25%,0.000000
50%,0.000000
75%,42.000000
max,511.000000


In [11]:
df_final['is_opposite_party'] = df_final['sponsor_party'] != df_final['cosponsor_party']

In [12]:
df_final.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party
0,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Foxx, Virginia [R-NC-5]",F000450,R,2025-01-13,HCONRES-3,2025-01-13,0,False
1,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Higgins, Clay [R-LA-3]",H001077,R,2025-01-13,HCONRES-3,2025-01-13,0,False
2,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Fleischmann, Charles J. ""Chuck"" [R-TN-3]",F000459,R,2025-01-13,HCONRES-3,2025-01-13,0,False
3,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Miller, Mary E. [R-IL-15]",M001211,R,2025-01-13,HCONRES-3,2025-01-13,0,False
4,119,3,HCONRES,House,2025-01-13,H11100,IntroReferral,Referred to the House Committee on Foreign Aff...,"Rep. Jackson, Ronny [R-TX-13]",J000304,R,"Rep. Aderholt, Robert B. [R-AL-4]",A000055,R,2025-01-13,HCONRES-3,2025-01-13,0,False


In [13]:
mask_cumple = (df_final['is_opposite_party'] == True) & (df_final['days_to_join'] == 0)
bills_validos = df_final[mask_cumple]['bill_code'].unique()

In [14]:
df_final_b = df_final[df_final['bill_code'].isin(bills_validos)].copy()

In [15]:
df_final_b.head()


,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,sponsor_party,cosponsor_name,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party
22,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Moore, Gwen [D-WI-4]",M001160,D,2025-01-28,HCONRES-4,2025-01-28,0,True
23,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Feenstra, Randy [R-IA-4]",F000446,R,2025-01-28,HCONRES-4,2025-01-28,0,False
24,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Sewell, Terri A. [D-AL-7]",S001185,D,2025-01-28,HCONRES-4,2025-01-28,0,True
25,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Carey, Mike [R-OH-15]",C001126,R,2025-01-28,HCONRES-4,2025-01-28,0,False
26,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,R,"Rep. Boyle, Brendan F. [D-PA-2]",B001296,D,2025-01-28,HCONRES-4,2025-01-28,0,True


In [16]:
df_final_b['credit'] = 0.2


# Day 0: 1.2 points
df_final_b.loc[(df_final_b['is_opposite_party'] == True) & (df_final_b['days_to_join'] == 0), 'credit'] = 1.2

# Day 1-5: 0.5 points
df_final_b.loc[(df_final_b['is_opposite_party'] == True) & (df_final_b['days_to_join'] > 0) & (df_final_b['days_to_join'] <= 5), 'credit'] = 0.5


# Day 0: 1.0 points
df_final_b.loc[(df_final_b['is_opposite_party'] == False) & (df_final_b['days_to_join'] == 0), 'credit'] = 1.0

# Day 1-5: 0.5 points
df_final_b.loc[(df_final_b['is_opposite_party'] == False) & (df_final_b['days_to_join'] > 0) & (df_final_b['days_to_join'] <= 5), 'credit'] = 0.5

In [17]:
weight_map = {
    'HR': 1.0, 'S': 1.0,
    'HJRES': 1.0, 'SJRES': 1.0,
    'HCONRES': 0.3, 'SCONRES': 0.3,
    'HRES': 0.1, 'SRES': 0.1
}

In [18]:
df_final_b['weight_type'] = df_final_b['Type'].map(weight_map).fillna(0)

In [19]:
df_final_b['weight_type'].value_counts()

,count
weight_type,
1.0,60415
0.1,12239
0.3,1014


In [20]:
df_final_b['total'] = df_final_b['weight_type'] * df_final_b['credit']

In [21]:
df_final_b.head()

,Congress,Number,Type,Origin_Chamber,Introduced_Date,latest_action_code,latest_action_type,latest_action_text,sponsor_name,sponsor_bioguide,...,cosponsor_bioguide,cosponsor_party,cosponsorship_date,bill_code,first_cosponsorshipDate,days_to_join,is_opposite_party,credit,weight_type,total
22,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,M001160,D,2025-01-28,HCONRES-4,2025-01-28,0,True,1.2,0.3,0.36
23,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,F000446,R,2025-01-28,HCONRES-4,2025-01-28,0,False,1.0,0.3,0.30
24,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,S001185,D,2025-01-28,HCONRES-4,2025-01-28,0,True,1.2,0.3,0.36
25,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,C001126,R,2025-01-28,HCONRES-4,2025-01-28,0,False,1.0,0.3,0.30
26,119,4,HCONRES,House,2025-01-28,H11100,IntroReferral,Referred to the House Committee on Ways and Me...,"Rep. LaHood, Darin [R-IL-16]",L000585,...,B001296,D,2025-01-28,HCONRES-4,2025-01-28,0,True,1.2,0.3,0.36


In [22]:
import numpy as np

# 1. Define the weight according to the bill type
weight_map = {
    'HR': 1.0, 'S': 1.0, 'HJRES': 1.0, 'SJRES': 1.0,
    'HCONRES': 0.3, 'SCONRES': 0.3, 'HRES': 0.1, 'SRES': 0.1
}

# 2. Identify the "best status" for each bill (did anyone cosponsor on day 0?)
# If it has at least one True on day 0, the whole bill is worth 1.0, otherwise 0.75
bill_status = (
    df_final_b[df_final_b['is_opposite_party'] == True]
    .groupby(['sponsor_bioguide', 'bill_code', 'Type'])
    .agg(tiene_dia_0=('days_to_join', lambda x: (x == 0).any()))
    .reset_index()
)

# 3. Assign the quality weight (1.0 or 0.75) based on that status
bill_status['score_calidad'] = np.where(bill_status['tiene_dia_0'], 1.0, 0.75)

# 4. Multiply by the bill-type weight
bill_status['factor_tipo'] = bill_status['Type'].map(weight_map).fillna(0)
bill_status['final_score'] = bill_status['score_calidad'] * bill_status['factor_tipo']

# 5. Group by sponsor to get the final total
cosponsor_summary = (
    bill_status.groupby('sponsor_bioguide')['final_score']
    .sum()
    .reset_index()
    .rename(columns={
        'sponsor_bioguide': 'primary_bioguide_id',
        'final_score': 'num_bills_with_cross_party_cosponsors'
    })
    .sort_values('num_bills_with_cross_party_cosponsors', ascending=False)
    .reset_index(drop=True)
)

cosponsor_summary.head(20)

,primary_bioguide_id,num_bills_with_cross_party_cosponsors
0,C001056,72.4
1,C001113,62.3
2,K000367,60.3
3,P000595,57.5
4,C001075,53.9
5,F000466,52.8
6,B001243,51.4
7,G000386,50.3
8,S001181,48.7
9,V000138,47.1


In [23]:
cosponsor_summary[cosponsor_summary['primary_bioguide_id']=='J000298']

,primary_bioguide_id,num_bills_with_cross_party_cosponsors
427,J000298,3.0


In [24]:
bill_status[bill_status['sponsor_bioguide']=='M001230']['final_score'].value_counts()

,count
final_score,
1.0,9


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
# Export
cosponsor_summary.to_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temp_Files/Temp_A/Temp_A_v1.4.0_2026_06_02.csv',
    index=False
)